# BÁO CÁO TIẾN ĐỘ LÀM VIỆC — NGÀY 2: HUẤN LUYỆN, TINH CHỈNH & TỐI ƯU HÓA MÔ HÌNH PHÂN LOẠI (MODEL TRAINING & TUNING)

## 🎯 MỤC TIÊU CỐT LÕI CỦA NGÀY 2
1. **Thiết lập mô hình cơ sở (Baseline Model)** bằng Hồi quy Logistic có `class_weight='balanced'` để làm mốc đối chuẩn cho bài toán phân loại `kem_tro_len`.
2. **Huấn luyện mô hình Cây quyết định (Decision Tree)**: Thiết lập cấu hình chống học vẹt (Overfitting), trích xuất cấu trúc luật cây trực quan phân tích các điều kiện thời tiết kích hoạt ô nhiễm.
3. **Huấn luyện mô hình Rừng ngẫu nhiên (Random Forest)**: Tận dụng cơ chế Bagging Ensemble, trích xuất thuộc tính `feature_importances_` để định lượng tầm quan trọng của các nhân tố (lớp biên pbl, hướng gió, ngày không mưa, khí phát thải CAMS).
4. **Huấn luyện mô hình Support Vector Machine (SVM)**: Triển khai trên dữ liệu scaled với kernel `rbf` và `linear`, cấu hình `probability=True` để phục vụ tính xác suất rủi ro.
5. **Huấn luyện mô hình nâng cao XGBoost (Gradient Boosting)**: Sử dụng tham số `scale_pos_weight` tối ưu hóa khả năng nhận diện mẫu thiểu số.
6. **Tinh chỉnh siêu tham số tự động (Hyperparameter Tuning)**: Sử dụng `GridSearchCV` với kiểm chứng chéo `TimeSeriesSplit` tối ưu theo `scoring='roc_auc'` hoặc `f1`.
7. **Đóng gói các mô hình tối ưu** vào file `.pkl` phục vụ đánh giá và bàn giao ở Ngày 3.

## 📋 CHI TIẾT CÁC BƯỚC THỰC HIỆN KÈM HƯỚNG DẪN & GIẢI THÍCH

### BƯỚC 2.1: Tái nạp dữ liệu và Thiết lập Baseline Model

In [12]:
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score

# 1. Nạp dữ liệu đã xử lý và chọn theo các đặc trưng
bundle = joblib.load(r"C:\Users\duong\projects\airvisual-data-collection\01_Data\tv2_data_doc\prepared_ml_bundle.pkl")
X_train_raw = bundle["X_train_raw"]
X_test_raw = bundle["X_test_raw"]
X_train_scaled = bundle["X_train_scaled"]
X_test_scaled = bundle["X_test_scaled"]
y_train = bundle["y_train"]
y_test = bundle["y_test"]
features_list = bundle["features_list"]

# 2. Huấn luyện Baseline Logistic Regression với class_weight='balanced'
baseline = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
baseline.fit(X_train_scaled, y_train)

y_pred_base = baseline.predict(X_test_scaled)
y_prob_base = baseline.predict_proba(X_test_scaled)[:, 1]

print("[BASELINE] LOGISTIC REGRESSION:")
print(f"  Accuracy : {accuracy_score(y_test, y_pred_base):.4f}")
print(f"  Recall   : {recall_score(y_test, y_pred_base, zero_division=0):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_base, zero_division=0):.4f}")
print(f"  F1-Score : {f1_score(y_test, y_pred_base, zero_division=0):.4f}")
print(f"  ROC-AUC  : {roc_auc_score(y_test, y_prob_base):.4f}")

[BASELINE] LOGISTIC REGRESSION:
  Accuracy : 0.9302
  Recall   : 1.0000
  Precision: 0.2500
  F1-Score : 0.4000
  ROC-AUC  : 0.9788


### BƯỚC 2.2: Huấn luyện Mô hình 1 — Decision Tree Classifier (Cây quyết định)
* **Ý nghĩa:** Giải thích minh bạch luật rẽ nhánh (Decision Rules).
* **Kiểm soát Overfitting:** Với lớp dương tính nhỏ (~7%), cần đặt `min_samples_leaf >= 5` để lá không bị học vẹt vào các điểm ngoại lai.

In [13]:
from sklearn.tree import DecisionTreeClassifier, export_text

dt_model = DecisionTreeClassifier(
    criterion="gini",
    max_depth=4,
    min_samples_split=20,
    min_samples_leaf=6,
    class_weight="balanced",
    random_state=42
)
dt_model.fit(X_train_raw, y_train)

tree_rules = export_text(dt_model, feature_names=features_list)
print("CẤU TRÚC LUẬT CỦA CÂY QUYẾT ĐỊNH (TOP NHÁNH):")
print(tree_rules[:800])

CẤU TRÚC LUẬT CỦA CÂY QUYẾT ĐỊNH (TOP NHÁNH):
|--- pm25_lag1 <= 34.25
|   |--- pbl_sang_m <= 68.93
|   |   |--- wind_rain_inter <= 214.15
|   |   |   |--- class: 0
|   |   |--- wind_rain_inter >  214.15
|   |   |   |--- class: 1
|   |--- pbl_sang_m >  68.93
|   |   |--- pm25_lag3 <= 56.65
|   |   |   |--- class: 0
|   |   |--- pm25_lag3 >  56.65
|   |   |   |--- class: 1
|--- pm25_lag1 >  34.25
|   |--- pm10_cams <= 28.69
|   |   |--- humidity <= 59.50
|   |   |   |--- class: 0
|   |   |--- humidity >  59.50
|   |   |   |--- class: 1
|   |--- pm10_cams >  28.69
|   |   |--- pm25_roll7_mean <= 33.36
|   |   |   |--- pm25_lag1 <= 44.25
|   |   |   |   |--- class: 0
|   |   |   |--- pm25_lag1 >  44.25
|   |   |   |   |--- class: 1
|   |   |--- pm25_roll7_mean >  33.36
|   |   |   |--- co_cams <= 654.08
|   |   |   |   |--- class: 0
|   |


### BƯỚC 2.3: Huấn luyện Mô hình 2 — Random Forest Classifier (Rừng ngẫu nhiên)
* **Ý nghĩa:** Tập hợp (Ensemble) nhiều cây quyết định độc lập giúp triệt tiêu phương sai, cung cấp khả năng tổng quát hóa vượt trội trên tập dữ liệu chuỗi thời gian.
* **Feature Importance:** Cung cấp bằng chứng định lượng trả lời Câu hỏi kinh doanh: Nhân tố nào tác động mạnh nhất tới nguy cơ ô nhiễm?

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=250,
    max_depth=6,
    min_samples_split=12,
    min_samples_leaf=4,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train_raw, y_train)

rf_importance = pd.DataFrame({
    "Feature": features_list,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT (RANDOM FOREST):")
print(rf_importance.head(10).to_string(index=False))

TOP 10 ĐẶC TRƯNG QUAN TRỌNG NHẤT (RANDOM FOREST):
         Feature  Importance
       pm25_lag1    0.148875
 pm25_roll3_mean    0.114104
 pm25_roll7_mean    0.094016
       pm10_cams    0.093601
       pm25_lag2    0.076417
        no2_cams    0.072152
stagnation_ratio    0.051508
            wind    0.042350
       pm25_lag3    0.039276
         co_cams    0.035278


### BƯỚC 2.4: Huấn luyện Mô hình 3 — Support Vector Machine (SVM)
* **Bắt buộc:**
  1. Dùng `X_train_scaled` và `X_test_scaled` (chuẩn hóa thang đo).
  2. Bật `probability=True` (để tính xác suất phân loại phục vụ ROC/PR curve).
  3. Cấu hình `class_weight='balanced'`.

In [15]:
from sklearn.svm import SVC

svm_rbf = SVC(
    kernel="rbf",
    C=1.5,
    gamma="scale",
    class_weight="balanced",
    probability=True,
    random_state=42
)
svm_rbf.fit(X_train_scaled, y_train)
print("Huấn luyện thành công mô hình SVM (RBF Kernel).")

Huấn luyện thành công mô hình SVM (RBF Kernel).


c:\Users\duong\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


### BƯỚC 2.5: Huấn luyện Mô hình 4 — XGBoost Classifier (Nâng cao)
* **Ý nghĩa:** XGBoost (Extreme Gradient Boosting) là chuẩn mực cho dữ liệu tabular với khả năng tối ưu hóa hàm mất mát có trọng số thông qua tham số `scale_pos_weight`.

In [16]:
from xgboost import XGBClassifier

# Tính tỷ lệ mất cân bằng giữa nhãn 0 và nhãn 1
ratio_imbalance = (len(y_train) - sum(y_train)) / sum(y_train)

xgb_model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=ratio_imbalance,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)
xgb_model.fit(X_train_raw, y_train)
print(f"Huấn luyện thành công XGBoost với scale_pos_weight = {ratio_imbalance:.2f}")

Huấn luyện thành công XGBoost với scale_pos_weight = 11.54


### BƯỚC 2.6: Tinh chỉnh Siêu tham số tự động (Hyperparameter Tuning với TimeSeriesSplit)
* Sử dụng `TimeSeriesSplit(n_splits=5)` để đảm bảo kiểm chứng chéo theo đúng dòng thời gian.
* Tiêu chí tối ưu: `scoring='roc_auc'` hoặc `scoring='f1'`.

In [17]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

cv_strategy = TimeSeriesSplit(n_splits=5)

# Lưới tham số cho Random Forest
param_grid_rf = {
    "n_estimators": [150, 250],
    "max_depth": [4, 6, 8],
    "min_samples_split": [8, 15]
}
grid_rf = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=42),
    param_grid_rf, cv=cv_strategy, scoring="roc_auc", n_jobs=-1
)
grid_rf.fit(X_train_raw, y_train)
best_rf = grid_rf.best_estimator_

# Lưới tham số cho SVM
param_grid_svm = {
    "C": [0.5, 1.0, 3.0],
    "gamma": ["scale", 0.05, 0.1]
}
grid_svm = GridSearchCV(
    SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=42),
    param_grid_svm, cv=cv_strategy, scoring="roc_auc", n_jobs=-1
)
grid_svm.fit(X_train_scaled, y_train)
best_svm = grid_svm.best_estimator_

print("\n KẾT QUẢ TINH CHỈNH TỐI ƯU:")
print("  Best Random Forest CV AUC:", round(grid_rf.best_score_, 4), grid_rf.best_params_)
print("  Best SVM (RBF)    CV AUC:", round(grid_svm.best_score_, 4), grid_svm.best_params_)

c:\Users\duong\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_search.py:1237: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan nan nan nan]
  warnings.warn(



 KẾT QUẢ TINH CHỈNH TỐI ƯU:
  Best Random Forest CV AUC: nan {'max_depth': 4, 'min_samples_split': 8, 'n_estimators': 150}
  Best SVM (RBF)    CV AUC: nan {'C': 0.5, 'gamma': 'scale'}


c:\Users\duong\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_search.py:1237: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan nan nan nan]
  warnings.warn(
c:\Users\duong\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


### BƯỚC 2.7: Đóng gói và Lưu trữ các mô hình tối ưu

In [18]:
models_bundle = {
    "baseline_logistic": baseline,
    "decision_tree": dt_model,
    "best_random_forest": best_rf,
    "best_svm": best_svm,
    "xgboost": xgb_model,
    "feature_importances": rf_importance
}

joblib.dump(models_bundle, r"C:\Users\duong\projects\airvisual-data-collection\01_Data\tv2_data_doc\trained_models_bundle.pkl")
print("Đã lưu trained_models_bundle.pkl thành công!")

Đã lưu trained_models_bundle.pkl thành công!
